# LAB 5 - Simulated Annealing
**Heating CLN025 to escape its minimum, and cooling it back**


Authors:

- Prof. Marco A. Deriu (marco.deriu@polito.it)
- Eric A. Zizzi (eric.zizzi@polito.it)
- Marcello Miceli (marcello.miceli@polito.it)

# Table of Contents

1. Simulated annealing
2. Annealing in GROMACS
3. The run
4. Running it
5. Did the temperature follow the schedule?
6. Melting on the way up, hysteresis on the way down
7. Back at 300 K: the same minimum?
8. How does the hairpin come apart?
9. Plain MD or annealing: which explores more?
10. Your deliverable

**Learning outcomes:**
- write a temperature schedule in a `.mdp` file and check that the thermostat followed it
- explain why annealing runs at constant volume once the water is above its boiling point
- recognise hysteresis, and say what it tells you about the time scales involved
- decide, from your own data, whether annealing found the native fold again

# 1. Simulated annealing

**Annealing** is the slow cooling of a molten material: given time at each temperature, the atoms settle into a well-ordered crystal, the state of lowest free energy.

**Simulated annealing** borrows the idea to search for low-energy states of complicated systems (Kirkpatrick, Gelatt & Vecchi, *Science* 1983, [doi:10.1126/science.220.4598.671](https://doi.org/10.1126/science.220.4598.671)). For a molecule, heat gives it the energy to cross the barriers between conformations; slow cooling lets it settle into a low-energy one:

- at high temperature the system visits high-energy regions and crosses high barriers;
- as the temperature drops, low-energy states become more and more probable (Boltzmann);
- only with infinitely slow cooling is the lowest state guaranteed.

That last point is the whole lesson of this lab. In practice the cooling is never slow enough, so annealing can only *suggest* a low-energy state, never prove it is the lowest. This lab runs the experiment once and asks you to judge the result.

<img src="imgs/simulated_annealing2.png" width="800">

# 2. Annealing in GROMACS

GROMACS anneals by changing the **reference temperature of the thermostat** over time. The schedule is a list of time points and temperatures, joined by straight lines, one list per temperature-coupling group:

| Option | Values | Meaning |
|---|---|---|
| `annealing` | `no`, `single`, `periodic` (one per group) | `single`: run the schedule once, then stay at its last temperature. `periodic`: start over after the last point |
| `annealing-npoints` | integer (one per group) | Number of points of the schedule |
| `annealing-time` | ps (npoints values per group) | Time of each point |
| `annealing-temp` | K (npoints values per group) | Reference temperature at each point |

The course couples the whole system as one group (`tc-grps = System`), so each option takes a single value or list.

<div class="alert alert-block alert-warning">
<b>The thermostat follows, it does not jump.</b> Only the reference temperature changes along the schedule; the system relaxes towards it with the thermostat's time constant (<code>tau-t</code> = 0.1 ps in the course). A schedule that changes faster than the system can follow is not obeyed. After a run, always check that the temperature did what you asked (section 5).
</div>

Because `single` holds the last temperature after the final point, a run whose `nsteps` goes beyond the schedule ends with a stretch at that temperature. This lab uses that on purpose: the schedule ends at 300 K and the run continues there, so the peptide has time to settle into whatever minimum it has landed in.

`grompp` prints the schedule it understood: read it.

# 3. The run

One run, starting from the end of the 300 K production run of LAB 3 (NMR model 1, AMBER) with new random velocities, at **constant volume**.

| Leg | Temperature | What it does |
|---|---|---|
| 1 | 300 → 400 K | warm past the melting point |
| 2 | 400 → 1100 K | melt it completely: no memory of the fold is left |
| 3 | 1100 → 600 K | the first, fast part of the cooling |
| 4 | 600 → 300 K | back to where we started |
| hold | 300 K | let it settle, and look at what we got |

Each leg lasts the same time, `leg_ns` below; the course's run uses 5 ns a leg, so 25 ns in all.

**Why constant volume?** Above 373 K, the boiling point of water at 1 bar, a barostat would let the box expand until the water became a dilute gas and the simulation meaningless. At constant volume the water keeps the density of the liquid: at 1100 K it is a hot, compressed fluid under a pressure of thousands of bar. That is nothing like physiological conditions; it is simply a fast and reliable way to erase the fold.

**Why 1100 K?** Not because it is physical, but because it has to be hot enough that the starting structure is forgotten within a leg. If it were not, the run would find its way back to the native fold for a trivial reason — it never really left.

Download the course's run in a terminal with `course-fetch --lab 05-SimulatedAnnealing --solutions`; it lands in `solutions/`, whole (trajectory of the entire system, energies, log) and with the peptide alone already prepared for the analysis.

## 3.1 The starting point

Choose where your LAB 3 run is, as in LAB 4 (`../03-ClassicalMD/solutions` for the course's):

In [ ]:
%env LAB3=../03-ClassicalMD

In [ ]:
%%bash
mkdir -p runs
cp $LAB3/cln025_model1/04-md/md_300K.gro runs/start.gro
cp $LAB3/cln025_model1/00-build/topol.top $LAB3/cln025_model1/00-build/posre.itp runs/
ls runs

## 3.2 The parameter file

The course's production parameters, with the barostat off, new velocities, and the schedule appended. Set `leg_ns` to what you can afford: the course's run uses 5 ns, and section 4 tells you how long that takes on your machine.

In [ ]:
import re

leg_ns = 5           # length of each leg, in ns; the course's run uses 5
hold_ns = leg_ns     # time at 300 K after the schedule ends

temperatures = [300, 400, 1100, 600, 300]
times = [round(i * leg_ns * 1000) for i in range(len(temperatures))]
total_ps = times[-1] + hold_ns * 1000

mdp = open("../common/mdp/md.mdp").read()
mdp = re.sub(r"(?m)^pcoupl .*", "pcoupl          = no        ; constant volume", mdp)
mdp = re.sub(r"(?m)^continuation .*", "continuation    = no", mdp)
mdp = re.sub(r"(?m)^gen-vel .*", "gen-vel         = yes\ngen-temp        = 300\ngen-seed        = -1", mdp)
mdp = re.sub(r"(?m)^nsteps .*", f"nsteps          = {round(total_ps / 0.002)}", mdp)
mdp += ("\n; Simulated annealing: " + " -> ".join(f"{T} K" for T in temperatures) +
        f", {leg_ns} ns a leg, then {hold_ns} ns at 300 K\n"
        "annealing               = single\n"
        f"annealing-npoints       = {len(temperatures)}\n"
        f"annealing-time          = {' '.join(str(t) for t in times)}\n"
        f"annealing-temp          = {' '.join(str(T) for T in temperatures)}\n")
with open("runs/anneal.mdp", "w") as f:
    f.write(mdp)
print(mdp[-300:])
print(f"total {total_ps / 1000:g} ns")

In [ ]:
%%bash
cd runs
gmx grompp -f anneal.mdp -c start.gro -p topol.top -o anneal.tpr > grompp.out 2>&1 || tail -n 20 grompp.out
grep -A 4 "Simulated annealing" grompp.out | head -n 8

# 4. Running it

Start it from a terminal, in the background, as in LAB 3:

```bash
cd ~/molbiomech/labs/05-SimulatedAnnealing/runs
nohup gmx mdrun -deffnm anneal -ntmpi 1 -v > anneal.out 2>&1 &
```

Use your ns/day from LAB 3 to work out how long it takes: at 25 ns the course's run is a long one, so if you cannot leave the machine going, set `leg_ns` to 1 and run 5 ns instead — the schedule is the same shape, only cooled five times faster. The analysis below works on your run, on the course's, or on both.

## 4.1 Keep the peptide

As in LAB 4, the analysis uses only the peptide. This prepares your run if it has finished (the course's run in `solutions/` is already prepared):

In [ ]:
%%bash
cd runs
if [ -f anneal.xtc ]; then
    printf "Protein\nProtein\n" | gmx trjconv -s anneal.tpr -f anneal.xtc -o anneal_protein.xtc -pbc mol -center > prep.out 2>&1
    printf "Protein\nProtein\n" | gmx trjconv -s anneal.tpr -f anneal.xtc -o anneal_protein.pdb -pbc mol -center -dump 0 >> prep.out 2>&1
    echo Protein | gmx convert-tpr -s anneal.tpr -o anneal_protein.tpr >> prep.out 2>&1
    echo "prepared runs/anneal"
else
    echo "no runs/anneal.xtc yet - the analysis will read solutions/anneal"
fi

Load the run, with the course's observables (the definitions of LAB 4, in `../common/scripts/cln025.py`) and the temperature of every frame. `RUN` picks your own run if you have one, otherwise the course's:

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import sys
import mdtraj as md

sys.path.append("../common/scripts")
import cln025
from cln025 import xvg

RUN = "runs/anneal" if os.path.exists("runs/anneal_protein.xtc") else "solutions/anneal"
if not os.path.exists(RUN + "_protein.xtc"):
    raise SystemExit("No annealing run to analyse yet. Either start yours (section 4) and run 4.1 when "
                     "it finishes, or fetch the course's in a terminal:\n"
                     "    course-fetch --lab 05-SimulatedAnnealing --solutions")

traj = md.load(RUN + "_protein.xtc", top=RUN + "_protein.pdb")
obs = cln025.observables(traj)
obs["time"] = traj.time                                       # ps
if not os.path.exists(RUN + ".temperature.xvg"):
    os.system(f"echo Temperature | gmx energy -f {RUN}.edr -o {RUN}.temperature.xvg > /dev/null 2>&1")
energy = xvg(RUN + ".temperature.xvg")
obs["T"] = np.interp(obs["time"], energy[:, 0], energy[:, 1])
folded = cln025.folded(obs)

print(f"{RUN}: {traj.n_frames} frames over {obs['time'][-1] / 1000:g} ns, "
      f"{obs['T'].min():.0f}-{obs['T'].max():.0f} K, folded in {folded.mean():.2f} of the frames")

# 5. Did the temperature follow the schedule?

The thermostat relaxes towards the reference temperature; it does not jump to it. Compare what you asked for with what you got:

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(obs["time"] / 1000, obs["T"], lw=0.6, label="the run")
ax.plot(np.array(times) / 1000, temperatures, "k--", marker="o", ms=4, label="schedule")
ax.axhline(300, color="gray", lw=0.6)
ax.set_xlabel("time (ns)")
ax.set_ylabel("temperature (K)")
ax.legend()
fig.tight_layout()
fig.savefig("schedule.png", dpi=300)
print(f"peak reached {obs['T'].max():.0f} K (asked for {max(temperatures)} K); "
      f"mean over the final hold {obs['T'][obs['time'] >= times[-1]].mean():.0f} K (asked for 300 K)")

<div class="alert alert-block alert-info">Does the measured temperature lag the schedule, and where most? A system of a few thousand atoms fluctuates by several kelvin around its reference even in equilibrium: how large are the fluctuations here compared with the 800 K the schedule climbs?</div>

# 6. Melting on the way up, hysteresis on the way down

Experimentally, CLN025 unfolds reversibly, like a two-state system, with a melting temperature of **70 °C (343 K)** (Honda, *Seibutsu Butsuri* 2009, [doi:10.2142/biophys.49.126](https://doi.org/10.2142/biophys.49.126); Honda et al., *J Am Chem Soc* 2008, [doi:10.1021/ja8030533](https://doi.org/10.1021/ja8030533)). That value is for CLN025 without caps, the molecule of the experiment.

Plot the fraction of native contacts Q against the temperature of each frame, separating the heating half of the run from the cooling half:

In [ ]:
peak = times[temperatures.index(max(temperatures))]           # time of the hottest point, in ps
heating = obs["time"] <= peak

fig, (left, right) = plt.subplots(1, 2, figsize=(12, 4.5))
left.plot(obs["T"][heating], obs["q"][heating], ".", ms=2, color="tab:red", label="heating")
left.plot(obs["T"][~heating], obs["q"][~heating], ".", ms=2, color="tab:blue", label="cooling")
left.axvline(343, color="gray", ls="--", lw=0.8, label="experimental $T_m$ (uncapped)")
left.set_xlabel("temperature (K)")
left.set_ylabel("Q")
left.legend(markerscale=5, fontsize=8)

right.plot(obs["time"] / 1000, obs["q"], lw=0.7, color="tab:purple")
twin = right.twinx()
twin.plot(obs["time"] / 1000, obs["T"], lw=0.6, color="tab:orange", alpha=0.6)
twin.set_ylabel("temperature (K)", color="tab:orange")
right.set_xlabel("time (ns)")
right.set_ylabel("Q", color="tab:purple")
fig.tight_layout()
fig.savefig("hysteresis.png", dpi=300)

opened = np.nonzero((obs["q"] < 0.5) & heating)[0]
print(f"on heating, Q first falls below 0.5 at {obs['T'][opened[0]]:.0f} K" if len(opened)
      else "on heating, Q never fell below 0.5")
# on the way down, a single frame above the threshold is noise: judge by the whole final hold
recovered = obs["q"][obs["time"] >= times[-1]].mean()
print(f"averaged over the final hold at 300 K, Q = {recovered:.2f}: "
      f"{'the fold came back' if recovered > 0.5 else 'the fold did not come back'}")

<div class="alert alert-block alert-info">
<ul>
<li>At what temperature does the hairpin open? How does it compare with the experimental melting temperature of 343 K, and with what LAB 4 said about this force field holding the fold too tightly?</li>
<li>The heating and cooling branches do not coincide: this is <b>hysteresis</b>, the signature of a process slower than the schedule. Folding CLN025 takes hundreds of nanoseconds or more — even at the highest temperature measured, its relaxation times are about 100 ns (Davis et al., <i>J Am Chem Soc</i> 2012, <a href="https://doi.org/10.1021/ja3046734">doi:10.1021/ja3046734</a>). Your cooling legs last a few nanoseconds. How much slower would they need to be?</li>
<li>Is the temperature at which Q collapses a melting temperature? A melting temperature is an equilibrium property: at it, the peptide must unfold <i>and refold</i> many times. What you measured is how hot it has to get before the fold gives way on this time scale: kinetics, not thermodynamics.</li>
</ul>
</div>

# 7. Back at 300 K: the same minimum?

The schedule begins and ends at 300 K, so the first and last stretches of the run are directly comparable: the same peptide, the same temperature, the same force field, with a trip to 1100 K in between. If annealing worked as a search, the peptide should be back in the native fold — or in something better.

Compare the two stretches, and the two structures:

In [ ]:
hold_start = times[-1]                                         # ps: the schedule ends, the hold at 300 K begins
window = min(1000, leg_ns * 1000)                              # ps at the start; never overlaps the hold
start = obs["time"] <= window
end = obs["time"] >= hold_start

print(f"{'':<16}{'T (K)':>7}{'e2e (nm)':>10}{'Q':>7}{'RMSD_NMR':>10}{'folded':>8}")
for label, mask in {f"first {window:g} ps": start, "the 300 K hold": end}.items():
    print(f"{label:<16}{obs['T'][mask].mean():7.0f}{obs['e2e'][mask].mean():10.2f}"
          f"{obs['q'][mask].mean():7.2f}{obs['rmsd'][mask].mean():10.3f}{folded[mask].mean():8.2f}")

# how far the final structure is from the one we started from
backbone = cln025.atoms(traj.topology, ("N", "CA", "C"))
moved = md.rmsd(traj[-1], traj[0], atom_indices=backbone)[0]
print(f"\nbackbone RMSD of the final structure from the starting one: {moved:.3f} nm")
print(f"starting structure: Q {obs['q'][0]:.2f}, RMSD from NMR {obs['rmsd'][0]:.3f} nm, "
      f"folded={bool(folded[0])}")
print(f"final structure:    Q {obs['q'][-1]:.2f}, RMSD from NMR {obs['rmsd'][-1]:.3f} nm, "
      f"folded={bool(folded[-1])}")

In [ ]:
traj[0].save("start_structure.pdb")
traj[-1].save("final_structure.pdb")
print("wrote start_structure.pdb and final_structure.pdb - open both in VMD and superimpose them")

Look at the two structures side by side:

```bash
vmd start_structure.pdb final_structure.pdb
```

with NewCartoon coloured by Structure, and Tyr1, Tyr2, Trp9 and Tyr10 as Licorice. In the Representations window, use `Extensions → Analysis → RMSD Trajectory Tool` to superimpose them.

<div class="alert alert-block alert-info">
<ul>
<li>Did the peptide come back to the fold it started from? If not, where did it end up — extended, collapsed but wrong, something with a helix?</li>
<li>The final state is a minimum the run fell into, not <i>the</i> minimum. What would you have to do to find out whether it is a deep one? (LAB 4 section 12 clustered the class dataset for exactly this reason.)</li>
<li>You cooled from 1100 K to 300 K in a few nanoseconds, a rate of order 10<sup>11</sup> K/s. Metallurgical annealing takes hours. Given the folding times above, is the failure to refold a statement about CLN025, about the force field, or about your schedule?</li>
<li>Run it again with a different <code>gen-seed</code>, or compare with a classmate's run. Do two runs of the same schedule end in the same place? What does that tell you about using a single annealing run as a structure-prediction method?</li>
</ul>
</div>

# 8. How does the hairpin come apart?

Okumura (2012) found that on heating, chignolin opens from the C-terminus, or from both ends at once, and may pass through an α-helix or 3₁₀-helix before unfolding completely. Check it on the heating half of your run: how much of the time is each of the three backbone hydrogen bonds of LAB 4 formed, when does it first break for at least 200 ps, and is there any helix?

In [ ]:
bonds = {"Asp3 N - Thr8 O": ((3, "N"), (8, "O")), "Gly7 N - Asp3 O": ((7, "N"), (3, "O")),
         "Thr8 N - Asp3 O": ((8, "N"), (3, "O")), "Tyr1 CA - Tyr10 CA": ((1, "CA"), (10, "CA"))}
cutoffs = {"Tyr1 CA - Tyr10 CA": 0.8}                          # open end-to-end: above 0.8 nm


def first_long_break(time, distance, cutoff, least_ps=200):
    """First time the distance stays above the cutoff for at least least_ps (on a smoothed curve)."""
    smooth = np.convolve(distance, np.ones(10) / 10, mode="same")
    needed = max(1, round(least_ps / (time[1] - time[0])))
    broken = 0
    for i, is_broken in enumerate(smooth >= cutoff):
        broken = broken + 1 if is_broken else 0
        if broken == needed:
            return time[i - needed + 1]
    return None


index = {(a.residue.resSeq, a.name): a.index for a in traj.topology.atoms}
print(f"{'bond':<22}{'formed (heating)':>18}{'first break':>14}")
for bond, (a, b) in bonds.items():
    cutoff = cutoffs.get(bond, 0.35)
    d = md.compute_distances(traj, [(index[a], index[b])])[:, 0]
    t = first_long_break(obs["time"], d, cutoff)
    formed = (d[heating] < cutoff).mean()
    print(f"{bond:<22}{100 * formed:17.0f}%{('never' if t is None else f'{t / 1000:.1f} ns'):>14}")

ss = md.compute_dssp(traj, simplified=False)[:, 1:-1]
print(f"\nframes with any helix (H, G or I): {np.isin(ss, ['H', 'G', 'I']).any(axis=1).mean():.2f}")

A bond counts as broken only if it stays broken for 200 ps: while the run is hot every bond flickers, and a single frame above the cut-off means nothing.

Watch it come apart in VMD, from a terminal in `runs` (or `solutions`):

```bash
vmd anneal_protein.pdb anneal_protein.xtc
```

with NewCartoon coloured by Structure, and Tyr1, Tyr2, Trp9 and Tyr10 as Licorice.

<div class="alert alert-block alert-info">Which bond is formed least of the time, and which breaks first? Does the C-terminal end (Thr8, Trp9, Tyr10) come apart before the N-terminal one, as Okumura found? Remember that Okumura simulated chignolin without caps, with a different AMBER force field, and at temperatures far below 1100 K.</div>

# 9. Plain MD or annealing: which explores more?

Clustering (LAB 4) counts how many distinct structures a run visits. Compare the 300 K run of LAB 3 with this one, with the same method and cut-off.

A run visits more structures simply by lasting longer, so the comparison is only fair on equal terms: the cell takes the same length of the 300 K run as your annealing run and clusters both at one frame every 100 ps. It prints how many structures went into each, so that you can check they match.

In [ ]:
%%bash -s "$total_ps"
mkdir -p clusters
src=$LAB3/cln025_model1/04-md/md_300K
anneal=$(ls runs/anneal_protein.xtc solutions/anneal_protein.xtc 2>/dev/null | head -n 1)
printf "Protein\nProtein\n" | gmx trjconv -s $src.tpr -f $src.xtc -o clusters/md_300K_protein.xtc -pbc mol -center -e $1 > clusters/prep.out 2>&1
echo Protein | gmx convert-tpr -s $src.tpr -o clusters/md_300K_protein.tpr >> clusters/prep.out 2>&1
for run in clusters/md_300K_protein ${anneal%.xtc}; do
    log=clusters/$(basename $run).log
    printf "Backbone\nProtein\n" | gmx cluster -s $run.tpr -f $run.xtc -dt 100 -method gromos -cutoff 0.1 \
        -g $log -cl clusters/$(basename $run).pdb > clusters/cluster.out 2>&1 || tail -n 20 clusters/cluster.out
    echo "$(basename $run): $(grep 'Found' $log) from $(awk '/structures for matrix/ {print $NF}' $log) structures"
done

<div class="alert alert-block alert-info">Why does the annealing run visit so many more structures? Most of them are unfolded: is "exploring more" the same as "finding the right structure"? Open the cluster middles (<code>clusters/*.pdb</code>) in VMD and see for yourself.</div>

# 10. Your deliverable

1. **The temperature check** of section 5: did your run follow its schedule, and where did it lag?
2. **The hysteresis plot** of section 6, with the temperature at which the hairpin opened and how it compares with the experimental 343 K.
3. **The verdict of section 7**: the two structures at 300 K, before and after, with the numbers and a VMD figure — did annealing bring the fold back? Answer in a short paragraph, and say what would change your mind.
4. **The unfolding pathway** of section 8, compared with Okumura (2012).

As for the other simulation labs, present them in a few slides during a 15-minute group discussion.